# 🦙 TSUM — Qwen2.5-14B unsloth 파인튜닝 (Instruction Tuning)

**목표**: `unsloth/Qwen2.5-14B-bnb-4bit` 모델을 크립토 뉴스/SNS 데이터로 파인튜닝해  
`bearish / neutral / bullish` 3-class 감성 분류기를 만든다.

| 항목 | 값 |
|------|----|
| 베이스 모델 | unsloth/Qwen2.5-14B-bnb-4bit |
| 양자화 | 4-bit NF4 (unsloth) |
| LoRA rank | r=16, alpha=32 |
| 타겟 모듈 | q/k/v/o/gate/up/down_proj |
| 학습 방식 | Instruction Tuning (SFTTrainer, ChatML) |
| 최대 토큰 길이 | 192 |
| GPU | RunPod (A100/H100/RTX 4090 등) |

> **RunPod**: GPU 인스턴스(A100 40GB 이상 권장)에서 JupyterLab으로 실행

---
## Step 1. 환경 설정

In [ ]:
import subprocess, sys

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✅ GPU 감지:', result.stdout.strip())
else:
    print('❌ GPU 없음 — 런타임 유형을 T4 GPU로 변경하세요')
    sys.exit()

In [ ]:
%%capture
# Unsloth + 필수 패키지 설치 (약 3~5분 소요)
!pip install unsloth
!pip install --upgrade trl datasets scikit-learn pandas numpy tqdm "wandb>=0.17.0" unsloth
print('✅ 패키지 설치 완료')

In [ ]:
import os, warnings, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm.auto import tqdm

from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset, load_dataset
from trl import SFTTrainer, DataCollatorForSeq2Seq
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings('ignore')
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'BF16 지원: {is_bfloat16_supported()}')

---
## Step 2. HuggingFace 로그인 (선택)

`unsloth/Qwen2.5-14B-bnb-4bit`는 **공개 모델**이라 로그인 없이 다운로드할 수 있습니다.  
(모델 업로드 또는 비공개 데이터셋 사용 시에만 필요)

In [ ]:
# 공개 모델이므로 로그인 불필요 — 업로드 시에만 아래 주석 해제
# from huggingface_hub import login
# login()
print('✅ Qwen2.5-14B-bnb-4bit 는 공개 모델 — HF 로그인 불필요')

---
## Step 2-b. Weights & Biases (W&B) 로그인

학습 중 loss·metric·LR 커브를 W&B 대시보드에서 실시간으로 확인합니다.  
https://wandb.ai/authorize 에서 API 키를 발급받으세요.

> **팁**: 이미 로그인된 환경(Colab Secrets 또는 `~/.netrc`)이면  
> `wandb.login()` 호출만으로 자동 인증됩니다.

In [ ]:
import wandb

# API 키 입력창이 뜨면 붙여넣고 Enter
# Colab Secrets에 WANDB_API_KEY를 등록했다면 자동 인증됩니다
wandb.login()
print('✅ W&B 로그인 완료')

---
## Step 3. 학습 설정 (하이퍼파라미터)

보고서 기반 최적 값으로 설정되어 있습니다. T4 VRAM 범위 내에서 조정 가능합니다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 모델 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BASE_MODEL   = 'unsloth/Qwen2.5-14B-bnb-4bit'
OUTPUT_DIR   = 'finetuned_qwen25_crypto'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# LoRA 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.1
TARGET_MODS   = ['q_proj', 'v_proj', 'k_proj', 'o_proj',
                 'gate_proj', 'up_proj', 'down_proj']

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 학습 하이퍼파라미터  (T4 16GB — 14B 모델 기준)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MAX_LENGTH    = 192
EPOCHS        = 3
BATCH_SIZE    = 8     # RunPod GPU (A100/H100) 기준
GRAD_ACCUM    = 4     # effective batch = 8 × 4 = 32
LR            = 2e-4
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
TEST_SIZE     = 0.1

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 클래스 / 라벨 설정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LABELS   = ['bearish', 'neutral', 'bullish']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

BULLISH_CUT =  1.5
BEARISH_CUT = -1.5
NOISE_CUT   =  0.3

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Qwen2.5 ChatML 시스템 프롬프트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SYSTEM_PROMPT = (
    "You are a crypto market sentiment analyzer. "
    "Classify the sentiment of the given text as exactly one of: "
    "bearish, neutral, bullish. Reply with only the label word."
)

print('✅ 설정 완료')
print(f'  Base model      : {BASE_MODEL}')
print(f'  Effective batch : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}')
print(f'  Label mapping   : {LABEL2ID}')

In [ ]:
import wandb, os

wandb.login()

WANDB_PROJECT  = 'tsum-qwen25-crypto-sentiment'
WANDB_RUN_NAME = f'qwen25-14b-r{LORA_R}-lr{LR}-ep{EPOCHS}'

os.environ['WANDB_PROJECT']   = WANDB_PROJECT
os.environ['WANDB_LOG_MODEL'] = 'checkpoint'
os.environ['WANDB_WATCH']     = 'gradients'

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    tags=['qwen2.5', 'unsloth', '14b', '4bit', 'instruction-tuning', 'crypto', 'sentiment'],
    config={
        'base_model':    BASE_MODEL,
        'lora_r':        LORA_R,
        'lora_alpha':    LORA_ALPHA,
        'lora_dropout':  LORA_DROPOUT,
        'target_modules': TARGET_MODS,
        'epochs':         EPOCHS,
        'batch_size':     BATCH_SIZE,
        'grad_accum':     GRAD_ACCUM,
        'effective_batch': BATCH_SIZE * GRAD_ACCUM,
        'learning_rate':  LR,
        'lr_scheduler':   'cosine',
        'warmup_ratio':   WARMUP_RATIO,
        'weight_decay':   WEIGHT_DECAY,
        'max_length':     MAX_LENGTH,
        'test_size':      TEST_SIZE,
        'labels':         LABELS,
        'bullish_cut':    BULLISH_CUT,
        'bearish_cut':    BEARISH_CUT,
        'noise_cut':      NOISE_CUT,
        'training_style': 'instruction-tuning',
        'quantization':   '4-bit NF4 (unsloth)',
        'bf16':           is_bfloat16_supported(),
        'optimizer':      'adamw_8bit',
    },
    resume='allow',
)
print(f'✅ W&B run 시작: {run.url}')
print(f'   Project : {WANDB_PROJECT}')
print(f'   Run name: {WANDB_RUN_NAME}')

---
## Step 4. 데이터셋 로드

### 사용 데이터셋 (총 약 6.3만 건)

| # | 출처 | 건수 | 내용 |
|---|------|------|------|
| A | `StephanAkkerman/financial-tweets-crypto` | ~41,000 | 크립토 트윗 + sentiment 라벨 |
| B | `SahandNZ/cryptonews-with-price-momentum-labels` | ~14,000 | 뉴스 헤드라인 + 3-class 모멘텀 라벨 |
| C | 커스텀 수집 (`collect_data.py`) | ~8,400 | CryptoPanic + Reddit 자동 라벨링 |

C는 로컬에서 `python training/collect_data.py --coins bitcoin ethereum solana dogecoin --days 60` 실행 후 `data/crypto_news_labeled.csv`를 업로드합니다.

In [ ]:
# ── 데이터셋 A: financial-tweets-crypto ──────────────────────────────────
print('📥 A: StephanAkkerman/financial-tweets-crypto 로드 중...')
ds_tweets = load_dataset('StephanAkkerman/financial-tweets-crypto', split='train')
df_tweets = ds_tweets.to_pandas()
print(f'   컬럼: {list(df_tweets.columns)}')
print(f'   건수: {len(df_tweets):,}')
df_tweets.head(2)

In [ ]:
# ── 데이터셋 B: cryptonews-with-price-momentum-labels ────────────────────
print('📥 B: SahandNZ/cryptonews-with-price-momentum-labels 로드 중...')
ds_news = load_dataset('SahandNZ/cryptonews-with-price-momentum-labels', split='train')
df_news = ds_news.to_pandas()
print(f'   컬럼: {list(df_news.columns)}')
print(f'   건수: {len(df_news):,}')
df_news.head(2)

In [ ]:
# ── 데이터셋 C: 커스텀 수집 데이터 (선택) ────────────────────────────────
# collect_data.py 로 생성한 CSV가 있으면 업로드, 없으면 건너뜁니다
df_custom = None

try:
    from google.colab import files
    print('▶ crypto_news_labeled.csv 파일을 선택하세요 (없으면 취소)')
    uploaded = files.upload()
    if uploaded:
        csv_path = list(uploaded.keys())[0]
        df_custom = pd.read_csv(csv_path)
        print(f'   업로드됨: {csv_path}  ({len(df_custom):,}건)')
    else:
        print('   건너뜀 — A·B 데이터만 사용합니다')
except Exception:
    print('   Colab 외부 환경 — 로컬 CSV를 직접 읽어보겠습니다')
    candidate = Path('data/crypto_news_labeled.csv')
    if candidate.exists():
        df_custom = pd.read_csv(candidate)
        print(f'   로컬 CSV 로드: {len(df_custom):,}건')

---
## Step 5. 전처리 및 데이터 병합

In [ ]:
# ── 텍스트 정규화 함수 ────────────────────────────────────────────────────
def clean_text(t: str) -> str:
    """URL 제거, 이모지·연속 공백 정규화. 대소문자 보존 (BTC ≠ btc)."""
    if not isinstance(t, str):
        return ''
    t = re.sub(r'http\S+|www\.\S+', '[URL]', t)           # URL
    t = re.sub(r'[^\x00-\x7F가-힣ㄱ-ㅎㅏ-ㅣ\s]', ' ', t)  # 이모지
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# ── 라벨 정규화 함수 ──────────────────────────────────────────────────────
LABEL_MAP = {
    # positive variants → bullish
    'positive': 'bullish', 'pos': 'bullish', 'buy': 'bullish',
    'bullish': 'bullish', 'up': 'bullish', '1': 'bullish', 1: 'bullish',
    # negative variants → bearish
    'negative': 'bearish', 'neg': 'bearish', 'sell': 'bearish',
    'bearish': 'bearish', 'down': 'bearish', '0': 'bearish', 0: 'bearish',
    # neutral
    'neutral': 'neutral', 'neu': 'neutral', 'hold': 'neutral',
    'none': 'neutral', '2': 'neutral', 2: 'neutral',
}

def normalize_label(val) -> str:
    if val is None:
        return 'neutral'
    key = str(val).lower().strip() if isinstance(val, str) else val
    return LABEL_MAP.get(key, 'neutral')

def price_to_label(change: float) -> str | None:
    """가격 변동 % → 라벨. 경계 노이즈(±0.3%) 샘플은 None 반환."""
    if abs(change) < NOISE_CUT:
        return None
    if change >= BULLISH_CUT:
        return 'bullish'
    if change <= BEARISH_CUT:
        return 'bearish'
    return 'neutral'

print('✅ 정규화 함수 정의 완료')

In [ ]:
parts = []

# ── A: financial-tweets-crypto ───────────────────────────────────────────
# 컬럼 자동 감지
text_col  = next((c for c in df_tweets.columns if c.lower() in ('text', 'tweet', 'content')), df_tweets.columns[0])
label_col = next((c for c in df_tweets.columns if c.lower() in ('sentiment', 'label', 'target')), None)

df_a = pd.DataFrame({'text': df_tweets[text_col].astype(str)})
if label_col:
    df_a['label'] = df_tweets[label_col].apply(normalize_label)
else:
    print('⚠️  A: 라벨 컬럼을 찾지 못해 neutral로 채웁니다')
    df_a['label'] = 'neutral'
df_a['source'] = 'financial_tweets'
parts.append(df_a)
print(f'A 로드: {len(df_a):,}건  (text_col={text_col}, label_col={label_col})')

# ── B: cryptonews-with-price-momentum-labels ──────────────────────────────
text_col2  = next((c for c in df_news.columns if c.lower() in ('headline', 'title', 'text', 'content')), df_news.columns[0])
label_col2 = next((c for c in df_news.columns if c.lower() in ('label', 'sentiment', 'momentum', 'price_label', 'target')), None)

df_b = pd.DataFrame({'text': df_news[text_col2].astype(str)})
if label_col2:
    df_b['label'] = df_news[label_col2].apply(normalize_label)
else:
    # price_change 컬럼으로 자동 라벨링 시도
    price_col = next((c for c in df_news.columns if 'price' in c.lower() or 'change' in c.lower()), None)
    if price_col:
        df_b['label'] = df_news[price_col].apply(lambda x: price_to_label(float(x)) or 'neutral')
    else:
        df_b['label'] = 'neutral'
df_b['source'] = 'cryptonews_momentum'
parts.append(df_b)
print(f'B 로드: {len(df_b):,}건  (text_col={text_col2}, label_col={label_col2})')

# ── C: 커스텀 데이터 (있을 때만) ──────────────────────────────────────────
if df_custom is not None:
    if 'price_change_24h' in df_custom.columns:
        df_custom['label'] = df_custom['price_change_24h'].apply(price_to_label)
        df_custom = df_custom.dropna(subset=['label'])
    df_c = pd.DataFrame({
        'text':   df_custom['text'].astype(str),
        'label':  df_custom['label'].apply(normalize_label),
        'source': 'custom_collected',
    })
    parts.append(df_c)
    print(f'C 로드: {len(df_c):,}건')

# ── 병합 및 공통 정제 ──────────────────────────────────────────────────────
df = pd.concat(parts, ignore_index=True)
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() >= 10]          # 너무 짧은 텍스트 제거
df = df.drop_duplicates('text')              # 중복 헤드라인 제거
df = df[df['label'].isin(LABELS)]            # 정규화 안 된 라벨 제거
df['label_id'] = df['label'].map(LABEL2ID)
df = df.reset_index(drop=True)

print(f'\n📊 병합 후 총 {len(df):,}건')
print(df['label'].value_counts())
df.head()

In [ ]:
# 클래스 불균형 확인 및 가중치 계산
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=df['label_id'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print('클래스 가중치 (balanced):')
for i, (lbl, w) in enumerate(zip(LABELS, class_weights)):
    cnt = (df['label_id'] == i).sum()
    print(f'  {lbl:8s} (id={i}): weight={w:.3f}  count={cnt:,}')

---
## Step 6. 토크나이저 & 데이터셋 준비

In [ ]:
# ── Qwen2.5 토크나이저 로드 ─────────────────────────────────────────────────
# unsloth는 FastLanguageModel.from_pretrained()로 모델과 토크나이저를 동시에 로드합니다.
# 여기서는 토큰 길이 분포만 확인합니다.
from transformers import AutoTokenizer

_tmp_tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

sample_lens = df['text'].sample(min(2000, len(df)), random_state=42).apply(
    lambda t: len(_tmp_tok.encode(t, add_special_tokens=False))
)
print(f'토큰 길이 분포 (샘플 {len(sample_lens):,}건):')
print(f'  평균: {sample_lens.mean():.1f}  중앙값: {sample_lens.median():.0f}')
print(f'  90%ile: {sample_lens.quantile(0.9):.0f}  99%ile: {sample_lens.quantile(0.99):.0f}')
print(f'  MAX_LENGTH={MAX_LENGTH} 기준 잘림 비율: {(sample_lens > MAX_LENGTH).mean()*100:.1f}%')
del _tmp_tok

In [ ]:
# ── Qwen2.5 ChatML 형식으로 변환 ─────────────────────────────────────────────
def make_conversation(text: str, label: str) -> str:
    """학습용: 전체 대화 (시스템 + 유저 + 어시스턴트 응답 포함)"""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{text}<|im_end|>\n"
        f"<|im_start|>assistant\n{label}<|im_end|>"
    )

def make_prompt(text: str) -> str:
    """추론용: 어시스턴트 응답 앞까지만 (모델이 이어서 생성)"""
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{text}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

df['conversation'] = df.apply(lambda r: make_conversation(r['text'], r['label']), axis=1)

# train/test 분할 (stratify by label)
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(
    df, test_size=TEST_SIZE, random_state=42, stratify=df['label']
)
df_train = df_train.reset_index(drop=True)
df_test  = df_test.reset_index(drop=True)

train_dataset = Dataset.from_dict({'text': df_train['conversation'].tolist()})
test_dataset  = Dataset.from_dict({
    'text':  df_test['text'].tolist(),
    'label': df_test['label'].tolist(),
})

print(f'Train: {len(df_train):,}  |  Test: {len(df_test):,}')
print('\n샘플 conversation:')
print(df_train['conversation'].iloc[0][:300], '...')

---
## Step 7. 모델 로드 (unsloth — Qwen2.5-14B-bnb-4bit)

unsloth는 표준 Hugging Face 대비 **2배 빠른 학습 + 60% 적은 VRAM**을 제공합니다.

T4 16GB VRAM 기준:
- Qwen2.5-14B (4-bit): ~7~8 GB
- LoRA 어댑터: ~0.3 GB
- 활성화 + 그래디언트 (batch=2, len=192): ~4~5 GB
- **총 예상: ~12~13 GB → T4에서 동작 가능**

In [ ]:
print(f'📥 모델 로드: {BASE_MODEL}  (약 5~10분 소요...)')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_LENGTH,
    dtype=None,          # auto: T4=fp16, A100=bf16
    load_in_4bit=True,
    trust_remote_code=True,
)

# Qwen2.5는 pad_token이 없음 → eos_token으로 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('✅ 모델 로드 완료')
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'   VRAM: {used:.1f} GB / {total:.1f} GB')

In [ ]:
# ── unsloth LoRA 적용 ─────────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODS,
    bias='none',
    use_gradient_checkpointing='unsloth',  # unsloth 최적화 버전
    random_state=42,
    use_rslora=False,
)
model.print_trainable_parameters()
# 예상: trainable params: ~107M || all params: 14.76B || trainable%: ~0.72%

---
## Step 8. 학습 (클래스 가중치 적용 Custom Trainer)

In [ ]:
from trl import SFTTrainer, DataCollatorForSeq2Seq

# SFTTrainer: instruction-tuning 전용 Trainer
# 응답 부분(어시스턴트 턴)에만 loss를 계산하고, 프롬프트는 마스킹합니다.
print('✅ SFTTrainer 준비 완료')

In [ ]:
def evaluate_generation(model, tokenizer, test_dataset, max_new_tokens=5, batch_size=16):
    """생성 방식으로 test set 전체를 평가합니다."""
    FastLanguageModel.for_inference(model)  # unsloth 추론 모드 활성화

    y_true, y_pred = [], []
    texts  = test_dataset['text']
    labels = test_dataset['label']

    for i in tqdm(range(0, len(texts), batch_size), desc='Evaluating'):
        batch_texts  = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]

        prompts = [make_prompt(t) for t in batch_texts]
        enc = tokenizer(
            prompts, return_tensors='pt', padding=True,
            truncation=True, max_length=MAX_LENGTH
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )
        # 프롬프트 이후 생성된 부분만 디코딩
        new_tokens = out[:, enc['input_ids'].shape[1]:]
        responses  = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

        for resp, true_label in zip(responses, batch_labels):
            resp = resp.strip().lower().split()[0] if resp.strip() else 'neutral'
            pred = resp if resp in LABELS else 'neutral'
            y_pred.append(pred)
            y_true.append(true_label)

    FastLanguageModel.for_training(model)  # 학습 모드로 복귀
    return y_true, y_pred

print('✅ evaluate_generation 함수 정의 완료')

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim='adamw_8bit',            # unsloth 권장 옵티마이저
    logging_steps=50,
    save_strategy='epoch',
    report_to='wandb',
    run_name=WANDB_RUN_NAME,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print('🚀 학습 시작 (RunPod 기준 epoch당 약 25~40분 소요 예상)...')
trainer.train()

---
## Step 9. 평가

In [ ]:
print('📊 Test set 평가 중 (생성 방식)...')
y_true, y_pred = evaluate_generation(model, tokenizer, test_dataset)

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
per_f1   = f1_score(y_true, y_pred, average=None, labels=LABELS, zero_division=0)

metrics = {
    'eval_accuracy':   accuracy,
    'eval_macro_f1':   macro_f1,
    'eval_bearish_f1': per_f1[0],
    'eval_neutral_f1': per_f1[1],
    'eval_bullish_f1': per_f1[2],
}

print('\n📈 최종 평가 결과 (Test set):')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Macro F1  : {macro_f1:.4f}')
print(f'  bearish F1: {per_f1[0]:.4f}')
print(f'  neutral F1: {per_f1[1]:.4f}')
print(f'  bullish F1: {per_f1[2]:.4f}')
print('\n--- 베이스라인 비교 ---')
print('  룰 기반          : macro_f1 = 0.41')
print('  CryptoBERT FT    : macro_f1 = 0.68')
print('  Qwen2.5-14B unsloth : macro_f1 = 0.74 (목표)')
print(f'  Qwen2.5-14B IFT  : macro_f1 = {macro_f1:.4f}  ← 현재')

In [ ]:
print(classification_report(y_true, y_pred, labels=LABELS, digits=4))

In [ ]:
# 빠른 추론 테스트 (unsloth 추론 모드)
test_texts = [
    'Bitcoin ETF inflows surge as institutional adoption accelerates',
    'Crypto market faces severe liquidation pressure amid regulatory crackdown',
    'Bitcoin trades sideways as market participants await Fed decision',
    'BTC 급등! 고래들 매집 중 — 강세 신호 포착',
    '암호화폐 해킹 사고로 대규모 자금 유출, 투자자 패닉',
]

FastLanguageModel.for_inference(model)
print('\n🧪 추론 테스트:')
for text in test_texts:
    prompt = make_prompt(text)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                       max_length=MAX_LENGTH).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=5, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip().lower()
    label = response.split()[0] if response.split() else 'neutral'
    label = label if label in LABELS else 'neutral'
    print(f'  [{label:8s}]  "{text[:65]}"')

---
## Step 10. 모델 저장 및 다운로드

다운로드한 zip 파일을 압축 해제해서 아래 위치에 넣으면 `inference.py`가 자동 감지합니다:

```
tsum/
└── models/
    └── finetuned_qwen25/       ← 여기
        ├── adapter_config.json     (PEFT 감지 키)
        ├── adapter_model.safetensors
        ├── config.json
        ├── tokenizer_config.json
        ├── tokenizer.model
        └── special_tokens_map.json
```

그리고 `config.yaml`의 `sentiment.model_path`를 `./models/finetuned_llama3`로 설정하세요.

In [ ]:
import shutil

# 최종 모델 저장 (LoRA 어댑터 + 토크나이저)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'✅ 저장 완료: {OUTPUT_DIR}/')
print('\n파일 목록:')
for f in sorted(Path(OUTPUT_DIR).iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:45s}  {size_mb:6.1f} MB')

In [ ]:
# zip 압축 후 Colab 다운로드
zip_path = f'{OUTPUT_DIR}.zip'
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
zip_size = Path(zip_path).stat().st_size / 1024 / 1024
print(f'압축 완료: {zip_path}  ({zip_size:.1f} MB)')

try:
    from google.colab import files
    files.download(zip_path)
    print('⬇️  다운로드 시작됩니다...')
except ImportError:
    print(f'Colab 외부: {zip_path} 를 직접 복사해 사용하세요')

In [ ]:
# ── W&B 최종 요약 ────────────────────────────────────────────────────────
# 평가 메트릭을 W&B Summary에 명시적으로 기록
wandb.summary.update({
    'final/accuracy':   metrics['eval_accuracy'],
    'final/macro_f1':   metrics['eval_macro_f1'],
    'final/bearish_f1': metrics['eval_bearish_f1'],
    'final/neutral_f1': metrics['eval_neutral_f1'],
    'final/bullish_f1': metrics['eval_bullish_f1'],
})

# 클래스별 F1 bar chart (W&B Table → Custom Chart)
f1_table = wandb.Table(
    columns=['class', 'f1_score'],
    data=[
        ['bearish', metrics['eval_bearish_f1']],
        ['neutral', metrics['eval_neutral_f1']],
        ['bullish', metrics['eval_bullish_f1']],
    ]
)
wandb.log({'class_f1_bar': wandb.plot.bar(f1_table, 'class', 'f1_score',
                                           title='Per-class F1 Score')})

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix([ID2LABEL[i] for i in y_true],
                      [ID2LABEL[i] for i in y_pred],
                      labels=LABELS)
wandb.log({
    'confusion_matrix': wandb.plot.confusion_matrix(
        probs=None,
        y_true=[ID2LABEL[i] for i in y_true],
        preds=[ID2LABEL[i] for i in y_pred],
        class_names=LABELS,
        title='Confusion Matrix (Test set)'
    )
})

# 모델 파일을 W&B Artifact로 업로드
artifact = wandb.Artifact(
    name=f'qwen25-qlora-crypto',
    type='model',
    description='Qwen2.5-14B unsloth 4-bit instruction-tuning — crypto sentiment (bearish/neutral/bullish)',
    metadata={'macro_f1': metrics['eval_macro_f1']}
)
artifact.add_dir(OUTPUT_DIR)
wandb.log_artifact(artifact)

wandb.finish()
print('✅ W&B 기록 완료')
print(f'   대시보드: {run.url}')

---
## (선택) Hugging Face Hub 업로드

Render/서버에서 모델을 직접 다운로드하고 싶을 때 사용합니다.  
업로드 후 `.env`에 `MODEL_PATH=your-hf-id/finetuned_qwen25_crypto` 설정.

In [ ]:
# HF_REPO = 'your-hf-username/finetuned_qwen25_crypto'
# trainer.model.push_to_hub(HF_REPO, commit_message='Qwen2.5-14B unsloth crypto sentiment')
# tokenizer.push_to_hub(HF_REPO)
# print(f'✅ 업로드 완료: https://huggingface.co/{HF_REPO}')
print('위 주석을 해제하고 HF_REPO를 본인 HuggingFace ID로 바꾼 뒤 실행하세요.')